# GIFs for Global Targeted Tasks
Generates two-panel animations for tasks with subtask input compressions of -0.1 and -0.2.

- **Panel 1**: Network with edges colored by $k_i^{-1}$ (icefire colormap, log-scale). Source nodes (top/bottom) and target nodes (left/right) highlighted.
- **Panel 2**: Strain response curve $(\varepsilon_\mathrm{in}, \varepsilon_\mathrm{out})$ with training-target diamonds and a red current-state marker.

In [4]:
import sys, json, io
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.collections import LineCollection
from matplotlib.colors import LogNorm
from matplotlib.ticker import LogLocator, LogFormatterSciNotation
import seaborn as sns
from PIL import Image


def _find_project_root():
    """Find the nonlinear_mech_nets root regardless of kernel CWD."""
    # VS Code sets __vsc_ipynb_file__ to the notebook's absolute path
    try:
        return Path(__vsc_ipynb_file__).resolve().parent.parent.parent  # noqa: F821
    except NameError:
        pass
    # Fall back: walk up from CWD looking for base/config.py
    p = Path('.').resolve()
    while p != p.parent:
        if (p / 'base' / 'config.py').exists():
            return p
        p = p.parent
    raise FileNotFoundError("Cannot find project root (expected base/config.py)")


ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from base.config import TARGETED_DATA_DIR, BOUNDARY_MARGIN
from analysis.data_io import load_auxetic_network
from analysis.trajectory import compute_auxetic_trajectory
from base.plot_config import apply_style

apply_style()
ICEFIRE = sns.color_palette('icefire', as_cmap=True)
VIDEOS_DIR = ROOT / 'videos'
VIDEOS_DIR.mkdir(exist_ok=True)

print(f'Project root:      {ROOT}')
print(f'Targeted data dir: {TARGETED_DATA_DIR}')
print(f'Output dir:        {VIDEOS_DIR}')


Project root:      /Users/fmartins/Library/CloudStorage/Box-Box/PhD/nonlinear_mech_nets
Targeted data dir: /Users/fmartins/Library/CloudStorage/Box-Box/PhD/nonlinear_mech_nets/data/auxetic_nets/targeted_results_sqr
Output dir:        /Users/fmartins/Library/CloudStorage/Box-Box/PhD/nonlinear_mech_nets/videos


In [5]:
# Discover tasks whose subtask compressions are exactly {-0.1, -0.2}
TARGET_COMPRESSIONS = {-0.2, -0.1}

qualifying = []  # list of (task_id, real_id, cfg)
for task_dir in sorted(Path(TARGETED_DATA_DIR).glob('task_*')):
    task_id = int(task_dir.name.split('_')[1])
    for real_dir in sorted(task_dir.glob('realization_*')):
        real_id = int(real_dir.name.split('_')[1])
        cfg_path = real_dir / 'task_config.json'
        if not cfg_path.exists():
            continue
        with open(cfg_path) as f:
            cfg = json.load(f)
        cs_set = set(cfg['compression_strains'])
        if cs_set == TARGET_COMPRESSIONS:
            qualifying.append((task_id, real_id, cfg))

print(f'Found {len(qualifying)} qualifying (task, real) pairs:')
for tid, rid, cfg in qualifying:
    print(f'  task_{tid:02d} / real_{rid:02d}  '
          f'strains={cfg["compression_strains"]}  '
          f'targets={cfg["target_poisson_ratios"]}')

Found 7 qualifying (task, real) pairs:
  task_01 / real_00  strains=[-0.2, -0.1]  targets=[-0.8, -0.8]
  task_02 / real_00  strains=[-0.2, -0.1]  targets=[-0.8, -1.0]
  task_03 / real_00  strains=[-0.2, -0.1]  targets=[-0.8, -0.4]
  task_04 / real_00  strains=[-0.2, -0.1]  targets=[-0.8, -0.6]
  task_05 / real_00  strains=[-0.2, -0.1]  targets=[-0.8, -0.3]
  task_06 / real_00  strains=[-0.2, -0.1]  targets=[-0.8, -0.5]
  task_07 / real_00  strains=[-0.2, -0.1]  targets=[-0.8, -0.2]


In [6]:
# --- Best-step stiffness loader --------------------------------------------

def load_best_stiffnesses(task_id, real_id, data_dir):
    """Load stiffnesses at the best (minimum-loss) training step."""
    base = Path(data_dir) / f'task_{task_id:02d}' / f'realization_{real_id:02d}'
    loss  = np.load(base / 'loss_trajectory.npy')
    stiff = np.load(base / 'stiffness_trajectory.npy')
    return stiff[np.argmin(loss)]


# --- Response curve from trajectory ----------------------------------------

def response_curve(traj, boundary):
    """Return (eps_in, nu) arrays for each trajectory step."""
    top, bottom = boundary['top'], boundary['bottom']
    left, right = boundary['left'], boundary['right']
    pos0 = np.array(traj[0])
    H0 = pos0[top, 1].mean() - pos0[bottom, 1].mean()
    W0 = pos0[right, 0].mean() - pos0[left, 0].mean()
    eps_in, nu = [], []
    for pos in traj:
        pos = np.array(pos)
        Ht = pos[top, 1].mean() - pos[bottom, 1].mean()
        Wt = pos[right, 0].mean() - pos[left, 0].mean()
        eps_yy = (Ht - H0) / H0
        eps_xx = (Wt - W0) / W0
        eps_in.append(max(0.0, -eps_yy))
        nu.append(-eps_xx / eps_yy if abs(eps_yy) > 1e-12 else 0.0)
    return np.array(eps_in), np.array(nu)


# --- Single frame -----------------------------------------------------------

NODE_COLORS = dict(top='#2ca02c', bottom='#d62728', left='#ff7f0e', right='#9467bd')
INTERIOR_COLOR = '#5b9bd5'

def make_frame(frame_idx, traj, network, boundary,
               eps_in_full, nu_full,
               compression_strains, target_poisson_ratios,
               inv_k, inv_k_vmin, inv_k_vmax,
               ref_positions):
    """Render one animation frame; returns a PIL Image."""
    positions = np.array(traj[frame_idx])
    edges = np.array(network.edges)
    top, bottom = boundary['top'], boundary['bottom']
    left, right = boundary['left'], boundary['right']
    boundary_mask = np.concatenate([top, bottom, left, right])
    interior = np.setdiff1d(np.arange(len(positions)), boundary_mask)

    fig = plt.figure(figsize=(13, 5.2), constrained_layout=True)
    gs = gridspec.GridSpec(1, 3, figure=fig,
                           width_ratios=[5.5, 0.25, 5.5], wspace=0.05)
    ax_net = fig.add_subplot(gs[0, 0])
    ax_cb  = fig.add_subplot(gs[0, 1])
    ax_rsp = fig.add_subplot(gs[0, 2])

    # ── Panel 1: network ────────────────────────────────────────────────────
    segments = [[positions[i], positions[j]] for i, j in edges]
    norm_log = LogNorm(vmin=inv_k_vmin, vmax=inv_k_vmax)
    lc = LineCollection(segments, cmap=ICEFIRE, norm=norm_log, linewidths=1.6, zorder=2)
    lc.set_array(inv_k)
    ax_net.add_collection(lc)

    ax_net.scatter(positions[interior, 0], positions[interior, 1],
                   c=INTERIOR_COLOR, s=22, zorder=3, linewidths=0)
    for key, idx in [('top', top), ('bottom', bottom), ('left', left), ('right', right)]:
        ax_net.scatter(positions[idx, 0], positions[idx, 1],
                       c=NODE_COLORS[key], s=50, zorder=4, linewidths=0.5,
                       edgecolors='k')

    ref = np.array(ref_positions)
    pad = 0.04
    dx = (ref[:, 0].max() - ref[:, 0].min()) * pad
    dy = (ref[:, 1].max() - ref[:, 1].min()) * pad
    ax_net.set_xlim(ref[:, 0].min() - dx, ref[:, 0].max() + dx)
    ax_net.set_ylim(ref[:, 1].min() - dy, ref[:, 1].max() + dy)
    ax_net.set_aspect('equal')
    ax_net.set_title(f'Strain = {eps_in_full[frame_idx]:.3f}', fontsize=13)
    ax_net.set_xticks([]); ax_net.set_yticks([])
    for sp in ax_net.spines.values():
        sp.set_visible(False)

    # ── Colorbar ────────────────────────────────────────────────────────────
    sm = plt.cm.ScalarMappable(cmap=ICEFIRE, norm=norm_log)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=ax_cb)
    cb.set_label(r'$k_i^{-1}$', fontsize=14)
    cb.ax.yaxis.set_major_locator(LogLocator(base=10, numticks=6))
    cb.ax.yaxis.set_major_formatter(LogFormatterSciNotation(base=10))
    cb.ax.tick_params(labelsize=10)

    # ── Panel 2: response curve ──────────────────────────────────────────────
    ax_rsp.plot(eps_in_full, nu_full, color='#1f77b4', linewidth=2.5, zorder=2)

    # Training-target diamonds (filled = subtask 0, open = subtask 1)
    target_labels = ['Strain target 1', 'Strain target 2']
    target_fills  = ['k', 'w']
    for s_idx, (cs, tp, lbl, fc) in enumerate(
            zip(compression_strains, target_poisson_ratios, target_labels, target_fills)):
        ax_rsp.scatter([-cs], [tp], marker='D', s=100,
                       c=fc, edgecolors='k', linewidths=1.5, zorder=5, label=lbl)

    # Current state
    ax_rsp.scatter([eps_in_full[frame_idx]], [nu_full[frame_idx]],
                   c='#d62728', s=80, zorder=6, label='Current state')

    ax_rsp.set_xlabel(r'$\varepsilon_{\rm in}$', fontsize=14)
    ax_rsp.set_ylabel(r'$\varepsilon_{\rm out}$', fontsize=14)
    ax_rsp.set_title('Strain response', fontsize=13)
    ax_rsp.tick_params(direction='in', which='both', top=True, right=True, labelsize=11)
    ax_rsp.legend(fontsize=10, framealpha=0.9, edgecolor='0.8')

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight', facecolor='white')
    buf.seek(0)
    img = Image.open(buf).copy()
    plt.close(fig)
    return img


In [ ]:
N_STEPS  = 200   # quasistatic steps — 200 needed for accurate Poisson ratio
TRAJ_TOL = 1e-8  # FIRE tolerance matching training convergence
FPS      = 20    # frames per second → duration_ms = 1000 // FPS
FRAME_MS = 1000 // FPS

import copy

for task_id, real_id, cfg in qualifying:
    cs_list = cfg['compression_strains']   # e.g. [-0.2, -0.1]
    tp_list = cfg['target_poisson_ratios']

    print(f'\n=== task_{task_id:02d} / real_{real_id:02d} ===')
    print(f'    compressions={cs_list}, targets={tp_list}')

    # Load network topology; override stiffnesses with best training step
    network, boundary = load_auxetic_network(task_id, real_id, data_dir=TARGETED_DATA_DIR)
    best_k = load_best_stiffnesses(task_id, real_id, data_dir=TARGETED_DATA_DIR)
    network = copy.copy(network)
    network.stiffnesses = best_k

    inv_k = 1.0 / best_k
    inv_k_vmin = max(inv_k.min() * 0.9, 1e-2)
    inv_k_vmax = inv_k.max() * 1.1

    # Trajectory to the LARGEST compression (first subtask = -0.2)
    max_cs = min(cs_list)
    print(f'    Computing trajectory (cs={max_cs}, n={N_STEPS}, tol={TRAJ_TOL})...',
          end=' ', flush=True)
    traj = compute_auxetic_trajectory(network, max_cs, boundary,
                                      n_steps=N_STEPS, tol=TRAJ_TOL)
    print('done.')

    eps_in, nu = response_curve(traj, boundary)
    print(f'    eps_in: [{eps_in.min():.4f}, {eps_in.max():.4f}]')
    print(f'    nu:     [{nu.min():.4f}, {nu.max():.4f}]')

    ref_positions = np.array(traj[0])

    # Render frames
    frames = []
    for fi in range(len(traj)):
        img = make_frame(
            fi, traj, network, boundary,
            eps_in, nu,
            cs_list, tp_list,
            inv_k, inv_k_vmin, inv_k_vmax,
            ref_positions,
        )
        frames.append(img)
        if fi % 50 == 0:
            print(f'    Frame {fi}/{len(traj)-1}', flush=True)

    # Save GIF
    out_path = VIDEOS_DIR / f'targeted_task{task_id:02d}_real{real_id:02d}.gif'
    frames[0].save(
        out_path,
        save_all=True,
        append_images=frames[1:],
        loop=0,
        duration=FRAME_MS,
        optimize=False,
    )
    print(f'    Saved → {out_path}')

print('\nAll done.')


In [ ]:
# Quick preview of the first task's first, mid, and last frame
from IPython.display import display
import copy

if qualifying:
    task_id, real_id, cfg = qualifying[0]
    network, boundary = load_auxetic_network(task_id, real_id, data_dir=TARGETED_DATA_DIR)
    best_k = load_best_stiffnesses(task_id, real_id, data_dir=TARGETED_DATA_DIR)
    network = copy.copy(network)
    network.stiffnesses = best_k

    inv_k = 1.0 / best_k
    inv_k_vmin = max(inv_k.min() * 0.9, 1e-2)
    inv_k_vmax = inv_k.max() * 1.1
    traj = compute_auxetic_trajectory(network, min(cfg['compression_strains']), boundary,
                                      n_steps=N_STEPS, tol=TRAJ_TOL)
    eps_in, nu = response_curve(traj, boundary)
    ref_positions = np.array(traj[0])

    for fi in [0, len(traj) // 2, len(traj) - 1]:
        img = make_frame(fi, traj, network, boundary, eps_in, nu,
                         cfg['compression_strains'], cfg['target_poisson_ratios'],
                         inv_k, inv_k_vmin, inv_k_vmax, ref_positions)
        display(img)
